# C2.7 · Benchmark design and critique

**Function C — Offensive Security & Research → The Security Researcher**  ·  *AI for Security*

---

**Risk.** Most published security benchmarks overstate real-world capability.

**Control.** Contamination checks; adapt methodology to your own corpus.

**This lab.** Contamination-check a public benchmark.

| | |
|---|---|
| Open-source tooling | Cyber Commons eval harness |
| Open-weight models | — |

> Runs anywhere: standard library only, no network, no API key. Where a lesson names a real tool you would deploy (Falco, OPA, SPIRE, Keycloak), the notebook models the *decision* that tool makes, so the lesson still lands on a machine that cannot pull containers.

In [ ]:
# --- Cyber Commons bootstrap -------------------------------------------------
# Puts the lab library on the path. Works from a clone, from the repo root, and
# on Kaggle. Standard library only — nothing to install, no network required.
import sys, os, subprocess
from pathlib import Path

def _find_labs():
    for base in [Path.cwd(), *Path.cwd().parents]:
        if (base / "labs" / "cybercommons" / "__init__.py").is_file():
            return base / "labs"
    # Kaggle kernels start in /kaggle/working with the repo absent. If the
    # kernel has internet enabled we clone it; if not, this raises and the
    # message tells you to attach the repo as a dataset instead.
    dest = Path("/kaggle/working/cyber-commons")
    if not dest.exists():
        subprocess.run(["git", "clone", "--depth", "1", "--branch", "claude/vulnbench-setup-scheduling-81aqov",
                        "https://github.com/spbreed/cyber-commons", str(dest)], check=True)
    return dest / "labs"

sys.path.insert(0, str(_find_labs()))
import cybercommons
print(cybercommons.banner("C2.7"))

Benchmark design and critique. Most security benchmarks fail on one of three things: class balance, held-out keys, or file matching.

In [ ]:
from cybercommons import evalkit

# failure 1 — class imbalance rewards guessing
skewed = {f"q{i}": evalkit.Truth(f"q{i}", "CWE-89", f"CWE-89/{i}.py") for i in range(1, 19)}
skewed.update({f"q{i}": evalkit.Truth(f"q{i}", "CWE-78", f"CWE-78/{i}.py") for i in (19, 20)})
lazy = {q: '{"qid":"%s","cwe":"CWE-89","file":"%s","rationale":"concatenated"}'
             % (q, t.file) for q, t in skewed.items()}
print("skewed benchmark, 'always guess CWE-89':")
print(evalkit.evaluate(lazy, skewed).render())

90% expert accuracy from a constant. Now balance the classes and re-run the identical strategy.

In [ ]:
balanced = {}
for i in range(1, 21):
    cwe = ["CWE-89", "CWE-78", "CWE-22", "CWE-798"][i % 4]
    balanced[f"q{i}"] = evalkit.Truth(f"q{i}", cwe, f"{cwe}/{i}.py")
lazy2 = {q: '{"qid":"%s","cwe":"CWE-89","file":"%s","rationale":"concatenated"}'
              % (q, t.file) for q, t in balanced.items()}
print("balanced benchmark, same strategy:")
print(evalkit.evaluate(lazy2, balanced).render())

The third failure is the file-matching bug from B2.10. A benchmark with all three problems can report any number its author prefers.

### Expect

The skewed benchmark gives the constant-guess strategy roughly 0.9 expert accuracy. The balanced one drops it to about 0.25 — with conformance at 1.0 in both cases.

### Your turn

Critique one public agentic-security benchmark against these three criteria. Write the critique as a repro card so someone can check your claim.

---

[All lessons](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/notebooks) · [Lesson page](https://spbreed.github.io/cyber-commons/lessons/C2.7.html) · [Lab library](https://github.com/spbreed/cyber-commons/tree/claude/vulnbench-setup-scheduling-81aqov/labs/cybercommons)

*Cyber Commons — a free, open commons for Cyber AI.*